# 05 Modelo Segmentacion Clientes

Este notebook transforma la base `EDA` a nivel cliente, entrena `K-Means` y exporta el `Parquet` final de segmentacion.


## 1. Librerias


In [ ]:
from pathlib import Path
import json

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 2. Definir rutas y cargar la base EDA


In [ ]:
from pathlib import Path

def resolve_project_root() -> Path:
    # Buscar la raiz del proyecto a partir del directorio actual.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "parquets").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto")

PROJECT_ROOT = resolve_project_root()
PROJECT_ROOT

# Definir las rutas de entrada y salida de la etapa 05.
INPUT_PATH = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets" / "02_base_eda_tickets.parquet"
OUTPUT_DIR = PROJECT_ROOT / "parquets" / "05_Modelo_Segmentacion_Clientes"
OUTPUT_PATH = OUTPUT_DIR / "05_clientes_segmentados.parquet"
MODEL_DIR = PROJECT_ROOT / "models" / "05_Modelo_Segmentacion_Clientes"
MODEL_PATH = MODEL_DIR / "05_modelo_kmeans.joblib"
FEATURES_PATH = MODEL_DIR / "05_features_kmeans.json"

# Leer la base EDA y preparar la fecha para calculos temporales.
df = pd.read_parquet(INPUT_PATH)
df["fecha"] = pd.to_datetime(df["fecha"])
df.head()


### Resultado breve

La etapa `05` ya no trabaja por ticket, sino que prepara una vista agregada por cliente para estudiar su comportamiento historico.


## 3. Construir la base por cliente


In [ ]:
# Obtener la fecha maxima para calcular recencia.
fecha_max = df["fecha"].max()

# Construir la base agregada a nivel cliente.
clientes = (
    df.groupby("id_cliente", dropna=False)
    .agg(
        numero_tickets=("id_ticket_modelado", "nunique"),
        gasto_total=("total_pedido", "sum"),
        ticket_promedio=("total_pedido", "mean"),
        ticket_maximo=("total_pedido", "max"),
        cantidad_total_productos=("cantidad_total", "sum"),
        promedio_productos_ticket=("cantidad_total", "mean"),
        platillos_distintos_totales=("platillos_distintos", "sum"),
        categorias_distintas_totales=("categorias_distintas", "sum"),
        ultimo_ticket=("fecha", "max"),
        primer_ticket=("fecha", "min"),
        ciudades_visitadas=("ciudad", "nunique"),
        sucursales_visitadas=("id_sucursal", "nunique"),
        usa_bebida=("incluye_bebida", "max"),
        usa_postre=("incluye_postre", "max"),
        usa_entrada=("incluye_entrada", "max"),
        usa_platillo_fuerte=("incluye_platillo_fuerte", "max"),
        metodo_pago_frecuente=("metodo_pago", lambda s: s.mode().iloc[0] if not s.mode().empty else "Sin dato"),
    )
    .reset_index()
)

# Derivar columnas de permanencia y recencia.
clientes["dias_activos"] = (clientes["ultimo_ticket"] - clientes["primer_ticket"]).dt.days + 1
clientes["dias_desde_ultimo_ticket"] = (fecha_max - clientes["ultimo_ticket"]).dt.days
clientes.head()


### Resultado breve

Con esta agregacion ya se observan clientes como unidades de analisis y no solo tickets individuales.


## 4. Seleccionar variables y entrenar el modelo


In [ ]:
# Definir las variables numericas para segmentacion.
FEATURES = [
    "numero_tickets",
    "gasto_total",
    "ticket_promedio",
    "ticket_maximo",
    "cantidad_total_productos",
    "promedio_productos_ticket",
    "categorias_distintas_totales",
    "dias_activos",
    "dias_desde_ultimo_ticket",
]

# Definir el preprocesador y el modelo K-Means.
preprocessor = ColumnTransformer(
    transformers=[("num", Pipeline(steps=[("scaler", StandardScaler())]), FEATURES)],
    remainder="drop",
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("clusterer", KMeans(n_clusters=3, random_state=42, n_init=20)),
])

# Entrenar el modelo de segmentacion.
pipeline.fit(clientes[FEATURES])
clientes["cluster"] = pipeline.predict(clientes[FEATURES])
clientes.head()


## 5. Nombrar segmentos y evaluar el clustering


In [ ]:
# Ordenar clusters por gasto y numero de tickets para asignar etiquetas.
ranking = (
    clientes.groupby("cluster")
    .agg(gasto_total_promedio=("gasto_total", "mean"), numero_tickets_promedio=("numero_tickets", "mean"))
    .sort_values(["gasto_total_promedio", "numero_tickets_promedio"])
    .reset_index()
)

etiquetas_base = [
    "Clientes de bajo valor",
    "Clientes de valor medio",
    "Clientes de alto valor",
]

mapping = {row["cluster"]: etiquetas_base[idx] for idx, (_, row) in enumerate(ranking.iterrows())}
clientes["segmento_cliente"] = clientes["cluster"].map(mapping)

# Calcular metricas basicas del clustering.
X_transformed = pipeline.named_steps["preprocessor"].transform(clientes[FEATURES])
metricas = pd.DataFrame(
    {
        "metrica": ["n_clientes", "n_clusters", "inercia", "silhouette"],
        "valor": [
            int(len(clientes)),
            3,
            round(float(pipeline.named_steps["clusterer"].inertia_), 4),
            round(float(silhouette_score(X_transformed, clientes["cluster"])), 4),
        ],
    }
)
metricas


### Resultado breve

Las metricas y las etiquetas permiten interpretar los grupos encontrados y revisar si la separacion de clientes es util para el negocio.


## 6. Visualizaciones del modelo


In [ ]:
# Graficar la cantidad de clientes por cluster y su comportamiento de gasto.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=clientes, x="cluster", ax=axes[0])
axes[0].set_title("Clientes por cluster")
axes[0].set_xlabel("cluster")
axes[0].set_ylabel("clientes")

sns.scatterplot(
    data=clientes,
    x="ticket_promedio",
    y="gasto_total",
    hue="cluster",
    palette="tab10",
    ax=axes[1],
)
axes[1].set_title("Ticket promedio vs gasto total")
axes[1].set_xlabel("ticket_promedio")
axes[1].set_ylabel("gasto_total")
plt.tight_layout()


### Resultado breve

Las graficas ayudan a ver el tamano relativo de cada cluster y su diferencia en terminos de gasto y comportamiento promedio.


## 7. Exportar el parquet final del modelo


In [ ]:
# Crear carpetas de salida para la etapa 05.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Exportar la base segmentada y los artefactos del modelo.
clientes.to_parquet(OUTPUT_PATH, index=False)
joblib.dump(pipeline, MODEL_PATH)
FEATURES_PATH.write_text(json.dumps(FEATURES, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Parquet generado en: {OUTPUT_PATH}")
print(f"Modelo guardado en: {MODEL_PATH}")


In [ ]:
# Verificar la salida exportada de la etapa 05.
clientes_exportados = pd.read_parquet(OUTPUT_PATH)
clientes_exportados.shape


### Resultado breve

La etapa `05` queda cerrada cuando el parquet final ya contiene el cluster y la etiqueta de segmento para cada cliente.
